# Acme Robotics walkthrough, notebook 2: data pipeline & operations

A runnable companion to `docs/ACME_ROBOTICS_WALKTHROUGH.md` sections 7-9 --
the CSV-to-RDF pipeline (`sketch` -> `triplify` -> `data`), the taxonomy-
membership check against real triplified data, and checking a *live*
triplestore instead of local files (`consistency-remote`). Same worked
example as notebook 1: `examples/acme_robotics/` (a small "Acme Robotics"
org chart built on the real W3C Organization Ontology and FOAF
vocabularies).

## 0. Setup

In [1]:
import subprocess, sys, os, tempfile, shutil, threading, json as _json
from pathlib import Path
import pandas as pd

def _find_repo_root(start):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file() and (candidate / "examples").is_dir():
            return candidate
    raise RuntimeError("could not find the repo root (looked for pyproject.toml + examples/) above " + str(start))

REPO_ROOT = _find_repo_root(Path.cwd())
os.chdir(REPO_ROOT)
EXAMPLE = "examples/acme_robotics"
assert (REPO_ROOT / EXAMPLE / "acme-org-v1.ttl").is_file(), \
    f"expected {EXAMPLE}/acme-org-v1.ttl under {REPO_ROOT}"
print("repo root:", REPO_ROOT)

LAST_EXIT_CODE = None

def run(*args, cwd=REPO_ROOT, check=True):
    """Runs the ontology-quality-suite CLI and prints its stdout. Returns
    nothing (keeps Jupyter's REPL-style auto-display from dumping a raw
    return value under every call); the exit code, if needed, is available
    afterwards as LAST_EXIT_CODE."""
    global LAST_EXIT_CODE
    proc = subprocess.run(["ontology-quality-suite", *args], cwd=cwd, capture_output=True, text=True)
    LAST_EXIT_CODE = proc.returncode
    print(proc.stdout)
    if proc.returncode != 0 and proc.stderr:
        print(proc.stderr, file=sys.stderr)
    if check and proc.returncode not in (0, 1):
        raise RuntimeError(f"exit {proc.returncode}")

def summary(out_dir):
    df = pd.read_csv(Path(out_dir) / "full_results.csv")
    return df.groupby(["check_id", "severity"]).size()

repo root: C:\repos\consolidated_ontology_suite_python


## 1. Static analysis before touching real data (`sketch`)

Walkthrough doc §7. Does `employees.rq` even reference vocabulary the
ontology declares? No CSV involved -- this is pure static analysis of the
query text, the cheapest possible signal.

In [2]:
run("sketch", "--queries", EXAMPLE, "--ontology", f"{EXAMPLE}/acme-org-v1.ttl",
    "--import-dir", f"{EXAMPLE}/reference_vocab", "--out-dir", "out/nb2-sketch", "--fail-on", "never")

Findings: 35 total (0 Violation, 0 Warning, 35 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb2-sketch
  - out\nb2-sketch\report.html (start here)
  - out\nb2-sketch\full_results.csv
Sketch used 1 query file(s); 8 triples, 7 entities.



## 2. Triplify (`triplify`)

Needs a built `oxi-gen` binary (see `config.find_oxi_gen_binary`) -- skips
gracefully below if one isn't found, the same pattern `docs/primer.ipynb`
uses for its own optional-dependency cells.

In [3]:
from ontology_suite import config

OXI_GEN_BIN = config.find_oxi_gen_binary()
print("oxi-gen binary:", OXI_GEN_BIN)
if OXI_GEN_BIN is None:
    print("no oxi-gen binary found (build it in a sibling ../oxi-gen checkout) -- skipping triplify+data below")
else:
    run("triplify", "--csv-dir", EXAMPLE, "--queries", EXAMPLE, "--out-dir", "out/nb2-triplify")

oxi-gen binary: C:\repos\oxi-gen\target\release\oxi_gen.exe


Triplified 1 file(s) into C:\repos\consolidated_ontology_suite_python\out\nb2-triplify
  - out\nb2-triplify\employees.ttl



## 3. Assess the real output, and a genuine fix verified against it (`data`)

Walkthrough doc §7. Checked against the real FOAF import, every
`foaf:name`/`foaf:mbox` triple used to report a `CNF-003`/`CNF-004`
domain/range "violation" -- not because Acme's data was wrong, but because
FOAF deliberately declares `foaf:name`'s `rdfs:domain` as `owl:Thing` (its
intentional "usable on absolutely anything" convention), and a naive
`rdfs:subClassOf` ancestor-walk can't discover that every class is
implicitly a subtype of `owl:Thing` under OWL semantics -- that's an
axiomatic fact, not an ordinary triple. `check_conformance` now
special-cases `owl:Thing`/`rdfs:Resource`/`rdfs:Literal` as always-satisfied
for exactly this reason; verified below against the real import, not a
synthetic approximation.

In [4]:
if OXI_GEN_BIN is not None:
    run("data", "out/nb2-triplify/employees.ttl", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl",
        "--import-dir", f"{EXAMPLE}/reference_vocab", "--engine", "sparql",
        "--own-namespace", "https://acme.example.org/",
        "--out-dir", "out/nb2-data", "--fail-on", "never")
    df = pd.read_csv("out/nb2-data/full_results.csv")
    foaf_domain_range = df[(df["check_id"].isin(["CNF-003", "CNF-004"])) & (df["path"].astype(str).str.contains("foaf", na=False))]
    print("foaf:name/foaf:mbox domain-range findings:", len(foaf_domain_range))
    assert len(foaf_domain_range) == 0, "owl:Thing/rdfs:Resource should be treated as always-satisfied"
    print("Confirmed: zero false positives against FOAF's owl:Thing-domain properties.")

Findings: 37 total (13 Violation, 22 Warning, 2 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb2-data
  - out\nb2-data\report.html (start here)
  - out\nb2-data\full_results.csv

foaf:name/foaf:mbox domain-range findings: 0
Confirmed: zero false positives against FOAF's owl:Thing-domain properties.


In [5]:
# Confirm the actual root cause directly: what does FOAF really declare?
import rdflib
g = rdflib.Graph().parse(f"{EXAMPLE}/reference_vocab/foaf.rdf", format="xml")
name = rdflib.URIRef("http://xmlns.com/foaf/0.1/name")
for s, p, o in g.triples((name, rdflib.RDFS.domain, None)):
    print(f"foaf:name rdfs:domain {o}  <- exactly the OWL2 universal class, not a blank-node union or anything more exotic")

foaf:name rdfs:domain http://www.w3.org/2002/07/owl#Thing  <- exactly the OWL2 universal class, not a blank-node union or anything more exotic


## 4. Closing the taxonomy gap with real data (`pattern-consistency --output-data`)

Notebook 1 §6 showed `pattern-consistency` reporting **clean** on the
static query+ontology+taxonomy check -- correctly, since `employees.rq`
builds its department reference dynamically per CSV row rather than
hard-coding it in the query text. Now that real triplified data exists,
passing it via `--output-data` closes exactly that gap:
`check_taxonomy_membership` checks values *actually used in the data*
against the taxonomy, not the query template. `employees.csv` row `E005`'s
department (`MKT`) isn't one of `taxonomy.ttl`'s three declared departments
(`ENG`/`QA`/`SALES`) -- this is exactly the failure mode a stale or
mistyped controlled-vocabulary reference in production data looks like:
valid IRI shape, no syntax error, just not real.

In [6]:
if OXI_GEN_BIN is not None:
    run("pattern-consistency",
        "--queries", f"{EXAMPLE}/employees.rq", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl",
        "--ontology", f"{EXAMPLE}/reference_vocab/org.ttl", "--ontology", f"{EXAMPLE}/reference_vocab/foaf.rdf",
        "--taxonomy", f"{EXAMPLE}/taxonomy.ttl", "--output-data", "out/nb2-triplify/employees.ttl",
        "--out-dir", "out/nb2-pattern-consistency")

== taxonomy <-> output data ==
  [undeclared_taxonomy_reference] https://acme.example.org/data/department/MKT is used as the value of https://acme.example.org/ns/worksIn in the data graph but is not declared as an individual anywhere in the given taxonomy set.

Written to: out\nb2-pattern-consistency\pattern-consistency.txt



## 5. Zero false positives against real triplified data (`ACM-001`)

Notebook 1 §3 demonstrated the project-specific `ACM-001` check firing
against a deliberately incomplete record. Here it runs against Acme's five
real, complete employee records instead -- confirming the earlier claim
that a project-specific check can be exercised against genuine data with
no false positives.

In [7]:
if OXI_GEN_BIN is not None:
    run("checks", "--ontology", f"{EXAMPLE}/acme-org-v1.ttl", "--data", "out/nb2-triplify/employees.ttl",
        "--registry", f"{EXAMPLE}/custom_checks/registry.json",
        "--sparql", f"{EXAMPLE}/custom_checks/sparql",
        "--engine", "sparql", "--out-dir", "out/nb2-acm001", "--fail-on", "never")
    acm001 = summary("out/nb2-acm001")
    print(acm001)
    assert acm001.empty, "the five real employee records all have acme:hasEmployeeId -- ACM-001 should not fire"
    print("Confirmed: zero false positives against Acme's real employee data.")

Findings: 0 total (0 Violation, 0 Warning, 0 Info)
Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb2-acm001
  - out\nb2-acm001\report.html (start here)
  - out\nb2-acm001\full_results.csv

Series([], dtype: int64)
Confirmed: zero false positives against Acme's real employee data.


## 6. Live triplestore checking (`consistency-remote`)

Walkthrough doc §9. Once Acme's org chart lives in a real triplestore
rather than Git-tracked Turtle, `consistency-remote` runs the same
three-way consistency check against it. No Docker/Fuseki install needed to
follow along here -- this cell starts a small, self-contained in-process
HTTP server implementing just enough of the SPARQL 1.1 Protocol for the
suite's `remote.fuseki` client to talk to.

This reuses the exact same in-process fake-Fuseki implementation this
repo's own test suite does (`tests/conftest.py`'s `fuseki_server` fixture,
exercised by `tests/test_fuseki.py`/`tests/test_cli_consistency_remote.py`)
-- just enough of the SPARQL 1.1 Protocol (`default-graph-uri`-scoped
`query`, plain `update`) for `remote.fuseki`'s real `urllib` request/response
code path, no mocking of `urllib` itself and no real Fuseki instance
needed.

In [8]:
import json as _json
import rdflib
from http.server import BaseHTTPRequestHandler, HTTPServer
from typing import cast
from urllib.parse import parse_qs
from rdflib.query import ResultRow as _ResultRow

EX = "https://acme.example.org/"
dataset = rdflib.Dataset()

class _Handler(BaseHTTPRequestHandler):
    def log_message(self, *a):
        pass

    def _dataset(self):
        return self.server.dataset

    def do_POST(self):
        length = int(self.headers.get("Content-Length", 0))
        body = self.rfile.read(length).decode("utf-8")
        params = parse_qs(body)

        if self.path.endswith("/update") or "update" in params:
            self._dataset().update(params["update"][0])
            self.send_response(200)
            self.end_headers()
            return

        query_text = params["query"][0]
        default_graphs = params.get("default-graph-uri", [])
        if default_graphs:
            merged = rdflib.Graph()
            for g in default_graphs:
                merged += self._dataset().get_context(rdflib.URIRef(g))
            result = merged.query(query_text)
        else:
            result = self._dataset().query(query_text, DEBUG=False)

        if result.type in ("CONSTRUCT", "DESCRIBE"):
            body_out = result.graph.serialize(format="turtle").encode("utf-8")
            self.send_response(200)
            self.send_header("Content-Type", "text/turtle")
            self.end_headers()
            self.wfile.write(body_out)
        else:
            bindings = []
            result_vars = result.vars or []
            for row in result:
                row = cast(_ResultRow, row)
                binding = {}
                for var in result_vars:
                    value = row[var]
                    if value is None:
                        continue
                    binding[str(var)] = {"type": "uri" if isinstance(value, rdflib.URIRef) else "literal", "value": str(value)}
                bindings.append(binding)
            payload = {"head": {"vars": [str(v) for v in (result.vars or [])]}, "results": {"bindings": bindings}}
            self.send_response(200)
            self.send_header("Content-Type", "application/sparql-results+json")
            self.end_headers()
            self.wfile.write(_json.dumps(payload).encode("utf-8"))

class _Server(HTTPServer):
    dataset: rdflib.Dataset

server = _Server(("127.0.0.1", 0), _Handler)
server.dataset = dataset
query_url = f"http://127.0.0.1:{server.server_address[1]}/query"
thread = threading.Thread(target=server.serve_forever, daemon=True)
thread.start()
print("fake SPARQL server listening at", query_url)

fake SPARQL server listening at http://127.0.0.1:49676/query


In [9]:
if OXI_GEN_BIN is not None:
    ontology_graph_uri = EX + "graph/ontology/1.0.0"
    data_graph_uri = EX + "graph/triplified/employees"

    ontology_ctx = dataset.get_context(rdflib.URIRef(ontology_graph_uri))
    ontology_ctx.parse(f"{EXAMPLE}/acme-org-v1.ttl", format="turtle")
    ontology_ctx.parse(f"{EXAMPLE}/reference_vocab/org.ttl", format="turtle")
    ontology_ctx.parse(f"{EXAMPLE}/reference_vocab/foaf.rdf", format="xml")

    data_ctx = dataset.get_context(rdflib.URIRef(data_graph_uri))
    data_ctx.parse("out/nb2-triplify/employees.ttl", format="turtle")

    manifest = {
        "graphs": [
            {"graph_uri": ontology_graph_uri, "role": "ontology"},
            {
                "graph_uri": data_graph_uri,
                "role": "triplified_data",
                "source_tarql": f"{EXAMPLE}/employees.rq",
                "ontology_graph_uri": ontology_graph_uri,
            },
        ]
    }
    manifest_path = Path(tempfile.mkdtemp()) / "manifest.json"
    manifest_path.write_text(_json.dumps(manifest), encoding="utf-8")

    run("consistency-remote", "--query-endpoint", query_url, "--manifest", str(manifest_path),
        "--out-dir", "out/nb2-consistency-remote")

=== https://acme.example.org/graph/triplified/employees ===
  source_tarql: examples/acme_robotics/employees.rq
  ontology_graph_uri: https://acme.example.org/graph/ontology/1.0.0
  template vs ontology: clean
  live data vs ontology: 0 undeclared class(es), 0 undeclared propert(y/ies)
  template vs live data: 0 discrepanc(y/ies) {'classes_only_in_template': [], 'classes_only_in_live_data': [], 'properties_only_in_template': [], 'properties_only_in_live_data': []}

Reports written to: C:\repos\consolidated_ontology_suite_python\out\nb2-consistency-remote



In [10]:
server.shutdown()
thread.join(timeout=5)
print("fake SPARQL server stopped.")

fake SPARQL server stopped.


---
## Next

- `docs/acme_robotics_lifecycle.ipynb` and `docs/ACME_ROBOTICS_WALKTHROUGH.md`
  cover ontology quality, versioning, and drift repair (§2-§6).
- `docs/MODELLING_PATTERN_CONSISTENCY.md` has the full four-boundary model
  behind §4's taxonomy-membership check.
- `docs/FUSEKI.md` has the full manifest spec and the `remote.fuseki` Python
  API for anything beyond the three-way consistency check demonstrated
  above.